# 003_postprocess_across_spatial_condition_specific.ipynb

Across-condition spatial postprocessing branch for the condition-specific decoding/ranking pipeline. Fits are loaded from notebook 001 outputs; decoding/ranking outputs are read from `msaa_condrank_decoding_outputs_*` where applicable.


In [ ]:

LOAD_DIR = "msaa_flexible_outputs_npz"
K_VALUES = [50, 14]

TOP_N_REPORT = 3
CLUSTER_RANGE = range(2, 9)

# Clustering-score tuning
PURITY_WEIGHT = 1.5
BALANCE_WEIGHT = 1
PENALTY_MODE = "weak_sqrt"   # "sqrt", "weak_sqrt", "linear", or "none"

NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

COND_COLORS = {0: "purple", 1: "green", 2: "black"}
COND_NAME_COLORS = {"intact": "purple", "word": "green", "rest": "black"}

SCHAEFER_TXT = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt"
SCHAEFER_NII = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
POSTERIOR_MAT = "data/pieman/raw/pieman_posterior_K700.mat"

OUTPUT_DIR = "003_postprocess_across_spatial_condition_specific_outputs"

In [ ]:
from pathlib import Path

%matplotlib inline
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.spatial.distance import cdist
from matplotlib.colors import ListedColormap, to_rgba
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    import nibabel as nib
    from nilearn.input_data import NiftiMasker
    from nilearn import plotting as niplot
    NILEARN_AVAILABLE = True
except Exception:
    NILEARN_AVAILABLE = False
    print("nilearn / nibabel not available; brain plotting disabled.")

print("Ready.")

In [ ]:
# ============================================================
# NOTEBOOK 3 IDENTITY / SPEED SETTINGS
# ============================================================

ANALYSIS_TYPE = "spatial"
FIT_SCOPE = "across"

USE_SAVED_DECODING = True
DECODING_OUTPUT_DIR = "msaa_condrank_decoding_outputs_spatial_across"
PER_ARCH_DECODING_CSV = os.path.join(DECODING_OUTPUT_DIR, "per_archetype_mean_accuracy.csv")

# Keep cache on so expensive clustering summaries do not recompute after first run.
USE_CACHE = True
OVERWRITE_CACHE = False


In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "003_postprocess_across_spatial"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)
_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None
    _fig_counter += 1
    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)
    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

def show_save_close(name=None):
    save_current_fig(name)
    plt.show()
    plt.close()

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
# ============================================================
# SAVED DECODING SUMMARY SETTINGS
# ============================================================

USE_SAVED_DECODING = True
DECODING_OUTPUT_DIR = "msaa_condrank_decoding_outputs_spatial_across"
PER_ARCH_DECODING_CSV = os.path.join(DECODING_OUTPUT_DIR, "per_archetype_mean_accuracy.csv")

def load_per_archetype_decoding(path=PER_ARCH_DECODING_CSV):
    if not os.path.exists(path):
        print("Saved per-archetype decoding CSV not found:", path)
        return pd.DataFrame()

    df = pd.read_csv(path)
    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "sem" not in df.columns:
        rename["sem_accuracy"] = "sem"
    if "err" in df.columns and "sem" not in df.columns:
        rename["err"] = "sem"
    df = df.rename(columns=rename)

    if "analysis_type" in df.columns:
        df = df[df["analysis_type"].astype(str) == "spatial"].copy()
    if "fit_scope" in df.columns:
        df = df[df["fit_scope"].astype(str) == "across"].copy()

    if "K" in df.columns:
        df["K"] = df["K"].astype(int)
    if "archetype" in df.columns:
        df["archetype"] = df["archetype"].astype(int)
    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str)

    print("Loaded saved per-archetype decoding:", path)
    print("Rows:", len(df))
    if len(df):
        display(df.head())
    return df

per_arch_decoding_df = load_per_archetype_decoding()

In [ ]:

# ============================================================
# CACHE SETTINGS
# ============================================================

from pathlib import Path
import pickle

CACHE_DIR = Path("003_postprocess_across_spatial_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set to True to use cached summaries/plot inputs if present.
USE_CACHE = True

# Set to True to recompute and overwrite cache.
OVERWRITE_CACHE = False

CLUSTER_SUMMARY_CACHE = CACHE_DIR / "cluster_summary_df.csv"
SELECTED_ARCHETYPES_CACHE = CACHE_DIR / "selected_archetypes_dict.npy"
PLOT_DATA_CACHE = CACHE_DIR / "plot_data_cache.pkl"

print("Cache directory:", CACHE_DIR.resolve())
print("USE_CACHE:", USE_CACHE)
print("OVERWRITE_CACHE:", OVERWRITE_CACHE)


def cache_exists(path):
    return Path(path).exists()


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

In [ ]:
# ============================================================
# LOAD SAVED PER-ARCHETYPE DECODING
# ============================================================

def load_per_archetype_decoding(path=PER_ARCH_DECODING_CSV):
    if not os.path.exists(path):
        print("Saved per-archetype decoding CSV not found:", path)
        return pd.DataFrame()

    df = pd.read_csv(path)
    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "sem" not in df.columns:
        rename["sem_accuracy"] = "sem"
    if "err" in df.columns and "sem" not in df.columns:
        rename["err"] = "sem"
    df = df.rename(columns=rename)

    if "analysis_type" in df.columns:
        df = df[df["analysis_type"].astype(str) == ANALYSIS_TYPE].copy()
    if "fit_scope" in df.columns:
        df = df[df["fit_scope"].astype(str) == FIT_SCOPE].copy()

    if "K" in df.columns:
        df["K"] = df["K"].astype(int)
    if "archetype" in df.columns:
        df["archetype"] = df["archetype"].astype(int)
    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str)

    print("Loaded saved per-archetype decoding:", path)
    print("Rows:", len(df))
    if len(df):
        display(df.head())
    return df

per_arch_decoding_df = load_per_archetype_decoding()


In [ ]:
def plot_decoding_bar_by_condition(results_subj, cond_labels, k, K):
    """
    Plot condition-specific single-archetype decoding from saved notebook-002 outputs.
    This avoids recomputing cross-validated decoding inside the postprocessing loop.
    """
    if "per_arch_decoding_df" not in globals() or len(per_arch_decoding_df) == 0:
        print(f"No saved decoding available for K={K}, archetype={k}; skipping decoding bar.")
        return

    dec_df = per_arch_decoding_df[
        (per_arch_decoding_df["K"].astype(int) == int(K)) &
        (per_arch_decoding_df["archetype"].astype(int) == int(k))
    ].copy()

    if len(dec_df) == 0:
        print(f"No saved decoding rows for K={K}, archetype={k}; skipping decoding bar.")
        return

    if "mean_accuracy" in dec_df.columns and "mean" not in dec_df.columns:
        dec_df = dec_df.rename(columns={"mean_accuracy": "mean"})
    if "sem_accuracy" in dec_df.columns and "sem" not in dec_df.columns:
        dec_df = dec_df.rename(columns={"sem_accuracy": "sem"})
    if "err" in dec_df.columns and "sem" not in dec_df.columns:
        dec_df = dec_df.rename(columns={"err": "sem"})

    order = [c for c in ["intact", "word", "rest"] if c in set(dec_df["condition"].astype(str))]
    dec_df["condition"] = pd.Categorical(dec_df["condition"].astype(str), categories=order, ordered=True)
    dec_df = dec_df.sort_values("condition")

    yerr = dec_df["sem"] if "sem" in dec_df.columns else None
    colors = []
    for c in dec_df["condition"].astype(str):
        if "COND_NAME_COLORS" in globals():
            colors.append(COND_NAME_COLORS.get(c, "gray"))
        else:
            colors.append("gray")

    plt.figure(figsize=(5, 4))
    plt.bar(dec_df["condition"].astype(str), dec_df["mean"], yerr=yerr, color=colors, capsize=4)
    plt.ylabel("Decoding accuracy")
    plt.title(f"K={K} | archetype {k} | saved single-archetype decoding")
    plt.tight_layout()
    show_save_close(f"saved_decoding_bar_K{K}_arch{k}")


In [ ]:

def to_float_array(x):
    return np.array(x, dtype=float)

def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()
    return {
        "K": int(data["K"]),
        "results_subj": results_subj,
        "condition_labels_str": data["condition_labels_str"].tolist(),
        "condition_codes": data["condition_codes"] if "condition_codes" in data else None,
        "condition_names": data["condition_names"].tolist() if "condition_names" in data else None,
    }

loaded = {}
for K in K_VALUES:
    path = os.path.join(LOAD_DIR, f"spatialAA_across_acrossCond_K{K}.npz")
    loaded[K] = load_msaa_npz(path)
    print("Loaded:", path)

In [ ]:

posterior = loadmat(POSTERIOR_MAT)
centers = np.asarray(posterior['posterior']['centers'][0][0][0][0][0], dtype=float)
widths = np.asarray(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T), dtype=float).ravel()

lookup_table = {
    'Vis': 'Visual',
    'SomMot': 'Somatomotor',
    'DorsAttn': 'Dorsal attention',
    'SalVentAttn': 'Ventral attention',
    'Limbic': 'Limbic',
    'Cont': 'Frontoparietal',
    'Default': 'Default mode'
}
network_colors = {
    'Visual': '#D7DF23',
    'Somatomotor': '#39B54A',
    'Dorsal attention': '#00A79D',
    'Ventral attention': '#27AAE1',
    'Limbic': '#1C75BC',
    'Frontoparietal': '#92278F',
    'Default mode': '#EE2A7B'
}
colors = ['#888888'] + [v for _, v in network_colors.items()]
network_cmap = ListedColormap(colors, N=len(colors) * 2, name='networks')
network_codes = {k: i + 1 for i, k in enumerate(lookup_table.values())}
print(network_codes)

In [ ]:

def nii2cmu(nifti_file, mask_file=None):
    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = nib.load(nifti_file) if type(nifti_file) == str else nifti_file
        mask = NiftiMasker(mask_strategy='background')
        mask.fit(nifti_file if mask_file is None else mask_file)

    S = img.get_sform()
    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]
    return {'Y': Y, 'R': R}

def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)
        label_weights = [sum(r[networks_cmu['Y'].ravel() == i]) for i in range(1, len(network_codes) + 1)]
        labels.append(np.argmax(label_weights) + 1)
    return pd.DataFrame({
        'code': labels,
        'Network': [list(lookup_table.values())[i - 1] for i in labels]
    })

key = pd.read_csv(
    SCHAEFER_TXT,
    sep='\t',
    header=None,
    names=['id', 'name', 'x', 'y', 'z', 't']
).drop('t', axis=1)
key['study'] = key['name'].apply(lambda x: x.split('_')[0])
key['hemisphere'] = key['name'].apply(lambda x: x.split('_')[1][0])
key['network'] = key['name'].apply(lambda x: x.split('_')[2])
key.drop('name', axis=1, inplace=True)
key['network'] = key['network'].apply(lambda x: lookup_table[x])
key['code'] = key['network'].apply(lambda x: network_codes[x])
key.set_index('id', inplace=True)
key.loc[0, 'code'] = 0

networks_cmu = nii2cmu(SCHAEFER_NII)
networks_cmu['Y'] = np.atleast_2d(np.array([key.loc[i, 'code'] for i in networks_cmu['Y']]).astype(float))
node_codes_template = node_labels(centers, widths, networks_cmu)
display(node_codes_template.head())

In [ ]:

def decoder(corrs):
    out = pd.DataFrame({'rank': [0.0], 'accuracy': [0.0], 'error': [0.0]})
    T = corrs.shape[0]
    for t in range(T):
        decoded_ind = int(np.argmax(corrs[t, :]))
        out.loc[0, 'error'] += np.mean(np.abs(decoded_ind - t)) / T
        out.loc[0, 'accuracy'] += (decoded_ind == t)
        out.loc[0, 'rank'] += np.mean((corrs[t, :] <= corrs[t, t]).astype(int))
    out['error'] /= T
    out['accuracy'] /= T
    out['rank'] /= T
    return out

def get_xval_assignments(ndata, nfolds, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))
    for i in range(1, nfolds):
        inds = np.arange(i * groupsize, min((i + 1) * groupsize, ndata))
        group_assignments[inds] = i
    rng.shuffle(group_assignments)
    return group_assignments

def build_single_archetype_recon_stack(subjects, k):
    Xhats = []
    for sub in subjects:
        sXC = np.asarray(sub["sXC"], dtype=float)
        S = np.asarray(sub["S"], dtype=float)
        Xhat = sXC[:, [k]] @ S[[k], :]
        Xhat = (Xhat - Xhat.mean(axis=0, keepdims=True)) / (Xhat.std(axis=0, keepdims=True) + 1e-8)
        Xhats.append(Xhat)
    return np.stack(Xhats, axis=0)

def run_timepoint_decoding(recon_stack, nfolds=2, nreps=20, seed=42):
    N, T, V = recon_stack.shape
    rng = np.random.default_rng(seed)
    all_results = []
    for rep in range(nreps):
        fold_ids = get_xval_assignments(N, nfolds, rng=rng)
        for i in range(nfolds):
            in_mask = (fold_ids == i)
            out_mask = ~in_mask
            in_mean = recon_stack[in_mask].mean(axis=0)
            out_mean = recon_stack[out_mask].mean(axis=0)
            corrs = 1.0 - cdist(in_mean, out_mean, metric='correlation')
            res = decoder(corrs)
            res["rep"] = rep
            res["fold"] = i
            all_results.append(res)
    return pd.concat(all_results, ignore_index=True)

def single_archetype_decoding_by_condition(results_subj, condition_labels_str, k, nfolds=2, nreps=20, seed=42):
    rows = []
    for cond_name in np.unique(condition_labels_str):
        idx = np.where(np.asarray(condition_labels_str) == cond_name)[0]
        sub_cond = [results_subj[i] for i in idx]
        recon_stack = build_single_archetype_recon_stack(sub_cond, k)
        res = run_timepoint_decoding(recon_stack, nfolds=nfolds, nreps=nreps, seed=seed)
        res["condition"] = cond_name
        res["archetype"] = k
        rows.append(res)
    return pd.concat(rows, ignore_index=True)

def summarize_single_archetype_decoding(results_subj, condition_labels_str, k, nfolds=2, nreps=20, seed=42):
    dec_df = single_archetype_decoding_by_condition(results_subj, condition_labels_str, k, nfolds=nfolds, nreps=nreps, seed=seed)
    summary = dec_df.groupby("condition")["accuracy"].agg(["mean", "std", "count"]).reset_index()
    summary["sem"] = summary["std"] / np.sqrt(summary["count"].clip(lower=1))
    summary["archetype"] = k
    overall = dec_df["accuracy"].mean()
    return dec_df, summary, overall

In [ ]:

def purity_score(y_cluster, y_class):
    y_cluster = np.asarray(y_cluster)
    y_class = np.asarray(y_class)
    total = 0
    for c in np.unique(y_cluster):
        mask = (y_cluster == c)
        _, counts = np.unique(y_class[mask], return_counts=True)
        total += counts.max()
    return total / len(y_cluster)

def equal_size_balance(labels):
    labels = np.asarray(labels)
    _, counts = np.unique(labels, return_counts=True)
    N = counts.sum()
    k = len(counts)
    ideal = N / k
    imbalance = np.sum(np.abs(counts - ideal)) / (2 * N)
    return float(1.0 - imbalance)

def cluster_count_penalty(n_clusters, mode="sqrt"):
    if mode == "linear":
        return 1.0 / n_clusters
    elif mode == "sqrt":
        return 1.0 / np.sqrt(n_clusters)
    elif mode == "weak_sqrt":
        return 1.0 / (n_clusters ** 0.25)
    elif mode == "none":
        return 1.0
    else:
        raise ValueError("mode must be 'linear', 'sqrt', 'weak_sqrt', or 'none'")

def score_clustering_solution(labels, cond, purity_weight=1.5, balance_weight=1.0, penalty_mode="sqrt"):
    labels = np.asarray(labels)
    cond = np.asarray(cond)
    n_clusters = len(np.unique(labels))
    purity = purity_score(labels, cond)
    balance = equal_size_balance(labels)
    penalty = cluster_count_penalty(n_clusters, mode=penalty_mode)
    score = (purity ** purity_weight) * (balance ** balance_weight) * penalty
    _, counts = np.unique(labels, return_counts=True)
    return {
        "n_clusters": n_clusters,
        "purity": purity,
        "balance": balance,
        "penalty": penalty,
        "score": score,
        "cluster_sizes": counts.tolist(),
    }

def get_timecourse_archetype_subject_matrix(results_subj, k, eps=1e-8):
    Xk = np.stack([np.asarray(sub["sXC"], dtype=float)[:, k] for sub in results_subj], axis=0)
    mu = Xk.mean(axis=1, keepdims=True)
    sd = Xk.std(axis=1, keepdims=True)
    Xk = (Xk - mu) / (sd + eps)
    return np.nan_to_num(Xk, nan=0.0, posinf=0.0, neginf=0.0)

def reorder_by_labels_and_within_similarity(sim, labels):
    labels = np.asarray(labels)
    unique_labels = np.unique(labels)
    order = []
    for lab in unique_labels:
        idx = np.where(labels == lab)[0]
        sub_sim = sim[np.ix_(idx, idx)]
        sub_order = idx[np.argsort(-sub_sim.mean(axis=1))]
        order.extend(sub_order.tolist())
    order = np.array(order)
    labels_sorted = labels[order]
    boundaries = np.where(np.asarray(labels_sorted[1:]) != np.asarray(labels_sorted[:-1]))[0] + 1
    return order, labels_sorted, boundaries

def make_result_from_labels(sim, labels, cond):
    order, labels_sorted, boundaries = reorder_by_labels_and_within_similarity(sim, labels)
    return {
        "sim": sim,
        "sim_sorted": sim[np.ix_(order, order)],
        "order": order,
        "labels": labels,
        "labels_sorted": labels_sorted,
        "cond_sorted": np.asarray(cond)[order],
        "boundaries": boundaries,
        "purity": purity_score(labels, cond),
        "ari": adjusted_rand_score(cond, labels),
        "nmi": normalized_mutual_info_score(cond, labels),
    }

def cluster_spectral(sim, cond, n_clusters=3, assign_labels="kmeans", random_state=0):
    sim = np.asarray(sim, dtype=float)
    sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)
    sim = np.clip(sim, -1.0, 1.0)
    aff = (sim + 1.0) / 2.0
    np.fill_diagonal(aff, 1.0)
    model = SpectralClustering(
        n_clusters=n_clusters,
        affinity="precomputed",
        assign_labels=assign_labels,
        random_state=random_state,
    )
    labels = model.fit_predict(aff) + 1
    return make_result_from_labels(sim, labels, cond)

def scan_spectral_cluster_numbers_for_archetype(Xk, sim, cond, cluster_range=range(2, 9), purity_weight=1.5, balance_weight=1.0, penalty_mode="sqrt", random_state=0):
    rows = []
    results = {}
    for n_clusters in cluster_range:
        res = cluster_spectral(sim=sim, cond=cond, n_clusters=n_clusters, assign_labels="kmeans", random_state=random_state)
        scored = score_clustering_solution(
            labels=res["labels"],
            cond=cond,
            purity_weight=purity_weight,
            balance_weight=balance_weight,
            penalty_mode=penalty_mode
        )
        rows.append({
            "n_clusters": n_clusters,
            "purity": scored["purity"],
            "balance": scored["balance"],
            "penalty": scored["penalty"],
            "score": scored["score"],
            "ari": res["ari"],
            "nmi": res["nmi"],
            "cluster_sizes": scored["cluster_sizes"],
        })
        results[n_clusters] = res
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True), results

In [ ]:

def plot_decoding_bar_from_summary(dec_summary, K, k):
    plot_df = dec_summary.copy()
    plt.figure(figsize=(5,4))
    plt.bar(
        plot_df["condition"],
        plot_df["mean"],
        yerr=plot_df["sem"],
        color=[COND_NAME_COLORS.get(c, "gray") for c in plot_df["condition"]],
        capsize=4
    )
    plt.ylabel("Decoding accuracy")
    plt.title(f"K={K} | archetype {k} | single-archetype decoding")
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()

def plot_cluster_scan_metrics(scan_df, archetype_k=None):
    plot_df = scan_df.sort_values("n_clusters")
    plt.figure(figsize=(8, 5))
    plt.plot(plot_df["n_clusters"], plot_df["purity"], marker="o", label="Purity")
    plt.plot(plot_df["n_clusters"], plot_df["balance"], marker="o", label="Balance")
    plt.plot(plot_df["n_clusters"], plot_df["score"], marker="o", label="Combined score")
    plt.xlabel("Number of clusters")
    plt.ylabel("Score")
    title = "Flexible spectral cluster-number scan"
    if archetype_k is not None:
        title += f" | archetype {archetype_k}"
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()

def plot_clustered_similarity(result, cond_colors=None, title="", show_subject_ids=True):
    if cond_colors is None:
        cond_colors = {0: "purple", 1: "green", 2: "black"}
    order = result["order"]
    cond_sorted = result["cond_sorted"]
    xticklabels = (order + 1) if show_subject_ids else False
    yticklabels = (order + 1) if show_subject_ids else False
    plt.figure(figsize=(7, 6))
    ax = sns.heatmap(
        result["sim_sorted"],
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        xticklabels=xticklabels,
        yticklabels=yticklabels,
        cbar_kws={"label": "subject correlation of archetype timecourses"},
    )
    if show_subject_ids:
        tick_colors = [cond_colors.get(int(c), "gray") for c in cond_sorted]
        for tick_label, color in zip(ax.get_xticklabels(), tick_colors):
            tick_label.set_color(color)
            tick_label.set_rotation(90)
            tick_label.set_fontsize(8)
        for tick_label, color in zip(ax.get_yticklabels(), tick_colors):
            tick_label.set_color(color)
            tick_label.set_fontsize(8)
    for b in result["boundaries"]:
        ax.axhline(b, color="black", linewidth=2)
        ax.axvline(b, color="black", linewidth=2)
    plt.title(f"{title}\nPur={result['purity']:.2f}, ARI={result['ari']:.2f}, NMI={result['nmi']:.2f}")
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()

def coefficient_to_alpha(weights, keep_mask, alpha_min=0.15, alpha_max=1.0):
    w = np.asarray(weights, dtype=float).ravel()
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    wk = w[keep_mask]
    if len(wk) == 0:
        return np.array([])
    wmin, wmax = wk.min(), wk.max()
    if np.isclose(wmax, wmin):
        return np.full(len(wk), alpha_max)
    scaled = (wk - wmin) / (wmax - wmin)
    return alpha_min + (alpha_max - alpha_min) * scaled

def plot_network_colored_spatial_coeff_map(results_subj, k, title="", display_mode="lyrz", node_size=10, thr_frac=0.30, use_opacity=False, alpha_min=0.15, alpha_max=1.0):
    if not NILEARN_AVAILABLE:
        print("nilearn not available; skipping.")
        return None, None, None
    coeffs = np.stack([np.asarray(sub["S"], dtype=float)[k, :] for sub in results_subj], axis=0)
    weights = np.nan_to_num(coeffs.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)
    wmax = np.max(weights)
    if wmax <= 0:
        print(f"Skipping archetype {k}: non-positive weights")
        return None, None, None
    keep = weights >= (thr_frac * wmax)
    centers_sel = centers[keep]
    node_codes_local = node_labels(centers, widths, networks_cmu)
    codes_sel = node_codes_local.loc[keep, 'code'].to_numpy()
    if use_opacity:
        alphas = coefficient_to_alpha(weights, keep, alpha_min=alpha_min, alpha_max=alpha_max)
        node_colors = []
        for code, alpha in zip(codes_sel, alphas):
            base = to_rgba(colors[code])
            node_colors.append((base[0], base[1], base[2], float(alpha)))
    else:
        node_colors = [colors[i] for i in codes_sel]
    display = niplot.plot_connectome(
        np.eye(centers_sel.shape[0]),
        centers_sel,
        node_size=node_size,
        node_color=node_colors,
        display_mode=display_mode,
        title=title,
    )
    save_current_fig()
    plt.show()
    plt.close()
    plt.clf()
    return display, node_codes_local, keep

def plot_network_pie_for_selected_nodes(node_codes_local, keep_mask, title="Network composition", out_file=None):
    selected = node_codes_local.loc[keep_mask].copy()
    counts = selected["Network"].value_counts().reindex(list(network_colors.keys()), fill_value=0)
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    pie_colors = [network_colors[name] for name in counts.index]
    ax.pie(
        counts.values,
        labels=counts.index,
        colors=pie_colors,
        autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
        startangle=90,
        counterclock=False,
    )
    ax.set_title(title)
    plt.tight_layout()
    save_current_fig()
    plt.show()
    plt.close()
    if out_file is not None:
        fig.savefig(out_file, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return counts

In [ ]:

def analyze_archetypes_for_K(cur, K_value, cluster_range=range(2, 9), purity_weight=1.5, balance_weight=1.0, penalty_mode="sqrt", nfolds=2, nreps=20, seed=42):
    results_subj = cur["results_subj"]
    condition_labels_str = np.array(cur["condition_labels_str"])
    cond = cur["condition_codes"]
    n_archetypes = np.asarray(results_subj[0]["sXC"]).shape[1]
    rows = []
    cluster_store = {}
    decoding_store = {}

    for k in range(n_archetypes):
        Xk = get_timecourse_archetype_subject_matrix(results_subj, k=k)
        sim = np.corrcoef(Xk)
        sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)
        sim = np.clip(sim, -1.0, 1.0)
        np.fill_diagonal(sim, 1.0)

        scan_df, scan_results = scan_spectral_cluster_numbers_for_archetype(
            Xk=Xk,
            sim=sim,
            cond=cond,
            cluster_range=cluster_range,
            purity_weight=purity_weight,
            balance_weight=balance_weight,
            penalty_mode=penalty_mode,
            random_state=0
        )

        best_cluster_row = scan_df.iloc[0]
        dec_df, dec_summary, dec_overall = summarize_single_archetype_decoding(
            results_subj,
            condition_labels_str,
            k,
            nfolds=nfolds,
            nreps=nreps,
            seed=seed,
        )

        row = {
            "K": K_value,
            "archetype": k,
            "best_n_clusters": int(best_cluster_row["n_clusters"]),
            "purity": float(best_cluster_row["purity"]),
            "balance": float(best_cluster_row["balance"]),
            "cluster_score": float(best_cluster_row["score"]),
            "ari": float(best_cluster_row["ari"]),
            "nmi": float(best_cluster_row["nmi"]),
            "cluster_sizes": best_cluster_row["cluster_sizes"],
            "decode_accuracy_overall": float(dec_overall),
        }
        for _, r in dec_summary.iterrows():
            cond_name = r["condition"]
            row[f"decode_{cond_name}"] = float(r["mean"])
            row[f"decode_{cond_name}_sem"] = float(r["sem"])

        rows.append(row)
        cluster_store[k] = {"scan_df": scan_df, "scan_results": scan_results}
        decoding_store[k] = {"dec_df": dec_df, "dec_summary": dec_summary}

    summary_df = pd.DataFrame(rows)
    summary_df["cluster_rank"] = summary_df["cluster_score"].rank(ascending=False, method="average")
    summary_df["decode_rank"] = summary_df["decode_accuracy_overall"].rank(ascending=False, method="average")
    summary_df["joint_rank_score"] = -(summary_df["cluster_rank"] + summary_df["decode_rank"])
    return {"summary_df": summary_df, "cluster_store": cluster_store, "decoding_store": decoding_store}

all_archetype_analysis = {}
all_rows = []
for K in K_VALUES:
    print(f"\n===== Analyzing archetypes for K={K} =====")
    cur = loaded[K]
    res = analyze_archetypes_for_K(
        cur,
        K_value=K,
        cluster_range=CLUSTER_RANGE,
        purity_weight=PURITY_WEIGHT,
        balance_weight=BALANCE_WEIGHT,
        penalty_mode=PENALTY_MODE,
        nfolds=NFOLDS_DECODE,
        nreps=NREPS_DECODE,
        seed=RNG_SEED
    )
    all_archetype_analysis[K] = res
    all_rows.append(res["summary_df"])
    display(
        res["summary_df"].sort_values(
            ["joint_rank_score", "cluster_score", "decode_accuracy_overall"],
            ascending=[False, False, False]
        ).head(10)
    )

archetype_summary_allK_df = pd.concat(all_rows, ignore_index=True)

In [ ]:

display(archetype_summary_allK_df.sort_values(["K", "joint_rank_score"], ascending=[True, False]))

best_archetype_rows = []
for K in K_VALUES:
    df = all_archetype_analysis[K]["summary_df"].copy()
    best_row = df.sort_values("joint_rank_score", ascending=False).iloc[0]
    best_archetype_rows.append(best_row)

best_archetypes_by_K_df = pd.DataFrame(best_archetype_rows)
display(best_archetypes_by_K_df)

In [ ]:

for K in K_VALUES:
    print("\n" + "="*90)
    print(f"Detailed plots for K={K}")
    print("="*90)

    cur = loaded[K]
    results_subj = cur["results_subj"]
    analysis_obj = all_archetype_analysis[K]
    top_df = analysis_obj["summary_df"].sort_values(
        ["joint_rank_score", "cluster_score", "decode_accuracy_overall"],
        ascending=[False, False, False]
    ).head(TOP_N_REPORT)

    for _, row in top_df.iterrows():
        k = int(row["archetype"])
        best_n_clusters = int(row["best_n_clusters"])
        best_cluster_result = analysis_obj["cluster_store"][k]["scan_results"][best_n_clusters]
        dec_summary = analysis_obj["decoding_store"][k]["dec_summary"]

        print(f"\nK={K} | archetype={k}")
        display(row.to_frame().T)
        display(dec_summary)

        plot_decoding_bar_from_summary(dec_summary, K, k)
        plot_cluster_scan_metrics(analysis_obj["cluster_store"][k]["scan_df"], archetype_k=k)
        plot_clustered_similarity(
            best_cluster_result,
            cond_colors=COND_COLORS,
            title=f"K={K} | archetype {k}",
            show_subject_ids=True
        )

        display1, node_codes_local, keep_mask = plot_network_colored_spatial_coeff_map(
            results_subj,
            k,
            title=f"K={K} | archetype {k} | network-colored coefficient plot",
            display_mode="lyrz",
            node_size=10,
            thr_frac=0.30,
            use_opacity=False
        )
        if display1 is not None:
            try:
                display1.close()
            except Exception:
                pass

        display2, node_codes_local2, keep_mask2 = plot_network_colored_spatial_coeff_map(
            results_subj,
            k,
            title=f"K={K} | archetype {k} | opacity-weighted coefficient plot",
            display_mode="lyrz",
            node_size=10,
            thr_frac=0.30,
            use_opacity=True,
            alpha_min=0.15,
            alpha_max=1.0
        )
        if display2 is not None:
            try:
                display2.close()
            except Exception:
                pass

        if node_codes_local is not None and keep_mask is not None:
            plot_network_pie_for_selected_nodes(
                node_codes_local,
                keep_mask,
                title=f"K={K} | archetype {k} | selected-node network composition"
            )

In [ ]:

archetype_summary_allK_df.to_csv(
    os.path.join(OUTPUT_DIR, "spatial_across_per_archetype_summary_clean_refactored.csv"),
    index=False
)
best_archetypes_by_K_df.to_csv(
    os.path.join(OUTPUT_DIR, "spatial_across_best_archetypes_by_K_clean_refactored.csv"),
    index=False
)
print("Saved summaries to:", OUTPUT_DIR)

## Cached clustering summary

In [ ]:

# ============================================================
# CLUSTERING SUBJECT MATRIX + SCAN HELPERS
# ============================================================

from sklearn.cluster import SpectralClustering

def zscore_rows(X, eps=1e-8):
    X = np.asarray(X, dtype=float)
    X = X - X.mean(axis=1, keepdims=True)
    X = X / (X.std(axis=1, keepdims=True) + eps)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


def get_clustering_subject_matrix(results_subj, analysis_type, k):
    """
    Returns subject x feature matrix for clustering a single archetype.

    Spatial AA:
        sXC is T x K, so archetype k is a timecourse.

    Temporal AA:
        sXC is V x K, so archetype k is a spatial motif.
    """
    rows = []
    for sub in results_subj:
        sXC = np.asarray(sub["sXC"], dtype=float)
        if k >= sXC.shape[1]:
            raise IndexError(f"Requested k={k}, but sXC has only {sXC.shape[1]} archetypes.")
        rows.append(sXC[:, k])

    return zscore_rows(np.vstack(rows))


def cluster_count_penalty(n_clusters, mode="sqrt"):
    if mode == "linear":
        return 1.0 / n_clusters
    elif mode == "sqrt":
        return 1.0 / np.sqrt(n_clusters)
    elif mode == "weak_sqrt":
        return 1.0 / (n_clusters ** 0.25)
    elif mode == "none":
        return 1.0
    else:
        raise ValueError("mode must be 'linear', 'sqrt', 'weak_sqrt', or 'none'")


def clustering_purity(labels, condition_codes):
    labels = np.asarray(labels)
    condition_codes = np.asarray(condition_codes)
    total = len(labels)
    score = 0

    for lab in np.unique(labels):
        idx = labels == lab
        _, counts = np.unique(condition_codes[idx], return_counts=True)
        score += counts.max()

    return score / total


def clustering_balance(labels):
    labels = np.asarray(labels)
    n = len(labels)
    labs = np.unique(labels)
    q = len(labs)
    sizes = np.array([(labels == lab).sum() for lab in labs], dtype=float)
    ideal = n / q
    return float(1.0 - (np.abs(sizes - ideal).sum() / (2 * n)))


def score_clustering_solution(
    labels,
    condition_codes,
    purity_weight=1.0,
    balance_weight=1.0,
    penalty_mode="sqrt"
):
    n_clusters = len(np.unique(labels))
    purity = clustering_purity(labels, condition_codes)
    balance = clustering_balance(labels)
    penalty = cluster_count_penalty(n_clusters, mode=penalty_mode)
    score = (purity ** purity_weight) * (balance ** balance_weight) * penalty

    cluster_sizes = {
        int(lab): int((labels == lab).sum())
        for lab in np.unique(labels)
    }

    return {
        "n_clusters": int(n_clusters),
        "purity": float(purity),
        "balance": float(balance),
        "penalty": float(penalty),
        "score": float(score),
        "cluster_sizes": cluster_sizes,
    }


def make_affinity_from_subject_matrix(X):
    X = zscore_rows(X)
    sim = np.corrcoef(X)
    sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)
    sim = np.clip(sim, -1.0, 1.0)

    aff = (sim + 1.0) / 2.0
    np.fill_diagonal(aff, 1.0)

    return aff, sim


def scan_cluster_numbers(
    X,
    condition_codes,
    cluster_range=None,
    purity_weight=1.0,
    balance_weight=1.0,
    penalty_mode="sqrt",
    random_state=0
):
    if cluster_range is None:
        cluster_range = CLUSTER_RANGE

    aff, sim = make_affinity_from_subject_matrix(X)
    rows = []

    for n_clusters in cluster_range:
        if n_clusters < 2 or n_clusters >= aff.shape[0]:
            continue

        labels = SpectralClustering(
            n_clusters=int(n_clusters),
            affinity="precomputed",
            assign_labels="kmeans",
            random_state=random_state,
        ).fit_predict(aff)

        scored = score_clustering_solution(
            labels,
            condition_codes,
            purity_weight=purity_weight,
            balance_weight=balance_weight,
            penalty_mode=penalty_mode,
        )

        rows.append({
            **scored,
            "labels": labels,
            "similarity": sim,
            "affinity": aff,
        })

    if len(rows) == 0:
        raise ValueError("No valid clustering solutions. Check CLUSTER_RANGE and subject count.")

    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

In [ ]:

# ============================================================
# CACHED CLUSTERING SUMMARY BUILDER
# ============================================================
#
# This cell is intended to replace repeated expensive clustering scans.
# It computes the per-archetype clustering summary once, saves it,
# and reloads it on future runs.

def compute_cluster_summary_cached():
    """
    Computes or loads the across-spatial clustering summary.

    Expected existing notebook variables/functions:
      loaded or fits
      K_VALUES
      ANALYSIS_TYPE
      get_clustering_subject_matrix
      scan_cluster_numbers
      CLUSTER_RANGE
      PURITY_WEIGHT
      BALANCE_WEIGHT
      PENALTY_MODE

    Output:
      cluster_summary_df with one row per K/archetype
    """

    if USE_CACHE and (not OVERWRITE_CACHE) and CLUSTER_SUMMARY_CACHE.exists():
        print("Loading cached cluster summary:", CLUSTER_SUMMARY_CACHE)
        return pd.read_csv(CLUSTER_SUMMARY_CACHE)

    rows = []

    for K in K_VALUES:
        print("\n" + "=" * 90)
        print(f"Computing cluster summary for K={K}")
        print("=" * 90)

        # Support either variable name from different notebook versions
        if "loaded" in globals():
            cur = loaded[K]
            results_subj = cur["results_subj"]
            cond_codes = cur["condition_codes"]
        elif "fits" in globals():
            cur = fits[K] if K in fits else fits.get(("across", K), None)
            if cur is None:
                raise KeyError(f"Could not find K={K} in fits.")
            results_subj = cur["results_subj"]
            cond_codes = cur["condition_codes"]
        else:
            raise NameError("Could not find `loaded` or `fits` containing MS-AA outputs.")

        n_archetypes = np.asarray(results_subj[0]["sXC"]).shape[1]

        for k in range(n_archetypes):
            Xk = get_clustering_subject_matrix(results_subj, globals().get("ANALYSIS_TYPE", "spatial"), k)
            scan_df = scan_cluster_numbers(
                Xk,
                cond_codes,
                cluster_range=CLUSTER_RANGE,
                purity_weight=PURITY_WEIGHT,
                balance_weight=BALANCE_WEIGHT,
                penalty_mode=PENALTY_MODE
            )
            best = scan_df.iloc[0]

            rows.append({
                "K": int(K),
                "archetype": int(k),
                "best_n_clusters": int(best["n_clusters"]),
                "purity": float(best["purity"]),
                "balance": float(best["balance"]),
                "cluster_score": float(best["score"]),
                "cluster_sizes": str(best.get("cluster_sizes", "")),
            })

    cluster_summary_df = pd.DataFrame(rows).sort_values(["K", "cluster_score"], ascending=[True, False])
    cluster_summary_df.to_csv(CLUSTER_SUMMARY_CACHE, index=False)
    print("Saved cluster summary:", CLUSTER_SUMMARY_CACHE)

    return cluster_summary_df


def select_archetypes_from_cluster_summary(cluster_summary_df, mode=None, top_n=None, threshold=None):
    """
    Selects archetypes from cached cluster summary.

    Uses notebook-level SELECTION_MODE, TOP_N_CLUSTER, CLUSTER_SCORE_THRESHOLD if not supplied.
    """
    if mode is None:
        mode = globals().get("SELECTION_MODE", "top_n")
    if top_n is None:
        top_n = globals().get("TOP_N_CLUSTER", 5)
    if threshold is None:
        threshold = globals().get("CLUSTER_SCORE_THRESHOLD", 0.35)

    selected = {}

    for K, sub in cluster_summary_df.groupby("K"):
        sub = sub.sort_values("cluster_score", ascending=False)

        if mode == "top_n":
            arches = sub.head(top_n)["archetype"].astype(int).tolist()
        elif mode == "threshold":
            arches = sub[sub["cluster_score"] >= threshold]["archetype"].astype(int).tolist()
            if len(arches) == 0:
                arches = [int(sub.iloc[0]["archetype"])]
        else:
            raise ValueError("mode must be 'top_n' or 'threshold'")

        selected[int(K)] = arches

    np.save(SELECTED_ARCHETYPES_CACHE, selected, allow_pickle=True)
    print("Saved selected archetypes:", SELECTED_ARCHETYPES_CACHE)

    return selected


cluster_summary_df = compute_cluster_summary_cached()
selected_archetypes_dict = select_archetypes_from_cluster_summary(cluster_summary_df)

display(cluster_summary_df.head(20))
print("Selected archetypes:")
for K, arches in selected_archetypes_dict.items():
    print(K, arches)

## Figure-only cache reload

In [ ]:

# ============================================================
# FIGURE-ONLY RERUN HELPER
# ============================================================
#
# After running the expensive cached summary once, future figure-only runs can:
#   1. set USE_CACHE = True
#   2. set OVERWRITE_CACHE = False
#   3. rerun from the loading cells onward
#
# If needed, this cell reloads cached selections directly.

if USE_CACHE and CLUSTER_SUMMARY_CACHE.exists():
    cluster_summary_df = pd.read_csv(CLUSTER_SUMMARY_CACHE)
    if SELECTED_ARCHETYPES_CACHE.exists():
        selected_archetypes_dict = np.load(SELECTED_ARCHETYPES_CACHE, allow_pickle=True).item()
    else:
        selected_archetypes_dict = select_archetypes_from_cluster_summary(cluster_summary_df)

    print("Reloaded cached cluster summary and selected archetypes.")
    display(cluster_summary_df.head())
    print(selected_archetypes_dict)
else:
    print("No cache found yet. Run the cached clustering summary cell first.")